# SETUP
---------------------------------------------------

In [17]:
import re
import nltk
import numpy as np
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



In [16]:
# Download the necessary datasets
nltk.download('punkt_tab')      # For word_tokenize
nltk.download('stopwords')  # For stopwords
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\M\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\M\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\M\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
df_train=pd.read_csv('xy_train.csv')
df_test=pd.read_csv('x_test.csv')
df_train


,ID,text,label
0,0,Americans Aren't Sure If Flight 370 Vanished T...,1
1,1,Pope Leo X offering indulgences to sinners sho...,0
2,2,News: 5 Uplifting Hypotheticals Of What Could ...,0
3,3,George W. Bush and Jeb Bush at a campaign stop...,0
4,4,The WADA requesting their fair share from Lanc...,0
...,...,...,...
47995,47995,"At Al-Aqsa Mosque Over 180,000 Muslims Prayed ...",1
47996,47996,"Texas Governor, Fearing USA Gov Invasion, Orde...",1
47997,47997,We are trying to teach high school students an...,1
47998,47998,Betty White accepts General Secretary of the C...,0


In [5]:
df_test

,ID,text
0,0,Local soccer club ditch shirt sponsorship deal...
1,1,What is this weird metal thing in the mail fro...
2,2,Donald Trump Jr reportedly told in email of Ru...
3,3,Satanist prayer is held in Alaskan government ...
4,4,There's A Spectacular Error In The Background ...
...,...,...
11950,11995,Asteroid Impact Warning: ESA Says This NEO Wil...
11951,11996,President Donald Trump (right) during his inte...
11952,11997,Two Jewish prisoners caught while attempting t...
11953,11998,Tool鈥檚 Maynard James Keenan Reveals Illness He...


The concatation is for preprocessing the train and test


* 0>--genuine 
* 1>--fake

# TEXT CLEANING
------------------------------------------------------------

In [81]:
def process_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\W', ' ', str(text))
    text = re.sub(r'\s[a-z]\s', ' ', text) 
    tokens = word_tokenize(text)
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    stop_words = set(stopwords.words("english"))
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [token for token in tokens if len(token) > 2]
    cleaned_text = " ".join(tokens)
    return cleaned_text

In [82]:
df_train['cleaned_text'] = df_train['text'].apply(process_text)
df_train

,text,label,cleaned_text,n_words
0,Americans Aren't Sure If Flight 370 Vanished T...,1,american sure flight 370 vanished thanks alien...,10
1,Pope Leo X offering indulgences to sinners sho...,0,pope leo offering indulgence sinner shortly pr...,10
2,News: 5 Uplifting Hypotheticals Of What Could ...,0,news uplifting hypothetical could happen banne...,13
3,George W. Bush and Jeb Bush at a campaign stop...,0,george bush jeb bush campaign stop father reel...,10
4,The WADA requesting their fair share from Lanc...,0,wada requesting fair share lance armstrong via...,12
...,...,...,...,...
47995,"At Al-Aqsa Mosque Over 180,000 Muslims Prayed ...",1,aqsa mosque 180 000 muslim prayed peacefully f...,10
47996,"Texas Governor, Fearing USA Gov Invasion, Orde...",1,texas governor fearing usa gov invasion order ...,13
47997,We are trying to teach high school students an...,1,trying teach high school student middle school...,31
47998,Betty White accepts General Secretary of the C...,0,betty white accepts general secretary communis...,15


In [11]:
df_train.isnull().sum()

text            0
label           0
cleaned_text    0
dtype: int64

In [12]:
df_train.duplicated(keep='last')

0        False
1        False
2        False
3        False
4        False
         ...  
47995    False
47996    False
47997    False
47998    False
47999    False
Length: 48000, dtype: bool

In [13]:
df_train['n_words'] = df_train['cleaned_text'].apply(lambda x: len(x.split()))
df_train['n_words'].describe()

count    48000.000000
mean        12.882875
std         22.006431
min          2.000000
25%         10.000000
50%         11.000000
75%         14.000000
max       1910.000000
Name: n_words, dtype: float64

## separate features and labels 

In [19]:
X= df_train['cleaned_text']
y = df_train['label']

In [21]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (38400,)
Shape of X_test: (9600,)
Shape of y_train: (38400,)
Shape of y_test: (9600,)


# TEXT REPRESENTATION
----------------------------------------------

In [ ]:
max_vocab = 20000
tokenizer = Tokenizer(num_words=max_vocab)
tokenizer.fit_on_texts(X_train)
vocabs = tokenizer.word_index 

vocab_size = len(vocabs) + 1
print("the size of vocab =", vocab_size)  

num_classes = len(np.unique(y))
print("the number of classes =", num_classes)

the size of vocab = 44619
the number of classes = 3


# vectorization
--------------------------------------

In [30]:
X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

# Padding
--------------------------------

In [31]:
maxlen = 40
X_train = pad_sequences(X_train, maxlen=maxlen)
X_test = pad_sequences(X_test, maxlen=maxlen)

# Build the model using RNN
---

In [57]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,Dense, Input, Dropout, SimpleRNN
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [58]:
emb_dim = 100
n_epochs = 6
batch_size = 32
learning_rate = 0.0001  

In [59]:
# Define the input layer with the specified sequence length
inputt = Input(shape=(maxlen,))

# Add an Embedding layer to learn word representations
x = Embedding(vocab_size, emb_dim)(inputt)

# Add an LSTM layer for sequence modeling
x = SimpleRNN(16)(x)
x = Dropout(0.5)(x)  # new line
# Add a Dense layer with ReLU activation
x = Dense(16, activation='relu')(x)
x = Dropout(0.5)(x)  # new line
# Add the output Dense layer with softmax activation for classification
x = Dense(3, activation='softmax')(x)

# Create the model
model_rnn = Model(inputt, x)
model_rnn.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_7 (Embedding)         │ (None, 40, 100)        │     4,461,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_7 (SimpleRNN)        │ (None, 16)             │         1,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,464,095 (17.03 MB)

 Trainable params: 4,464,095 (17.03 MB)

 Non-trainable params: 0 (0.00 B)

In [60]:
# Define optimizer with specified learning rate
optimizer = Adam(learning_rate=learning_rate)
model_rnn.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [61]:
history_rnn = model_rnn.fit(X_train, y_train,
                            epochs=n_epochs, batch_size=batch_size,
                            validation_data=(X_test, y_test))

Epoch 1/6
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 115s 94ms/step - accuracy: 0.5158 - loss: 0.8756 - val_accuracy: 0.6032 - val_loss: 0.7238
Epoch 2/6
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 113s 94ms/step - accuracy: 0.5739 - loss: 0.7500 - val_accuracy: 0.6647 - val_loss: 0.6475
Epoch 3/6
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 128s 106ms/step - accuracy: 0.6614 - loss: 0.6465 - val_accuracy: 0.7642 - val_loss: 0.5567
Epoch 4/6
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 114s 95ms/step - accuracy: 0.7658 - loss: 0.5324 - val_accuracy: 0.7789 - val_loss: 0.4913
Epoch 5/6
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 128s 107ms/step - accuracy: 0.8391 - loss: 0.4250 - val_accuracy: 0.7868 - val_loss: 0.4717
Epoch 6/6
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 122s 102ms/step - accuracy: 0.8840 - loss: 0.3343 - val_accuracy: 0.7835 - val_loss: 0.5004


In [63]:
loss, accuracy = model_rnn.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)


300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7835 - loss: 0.5004
Test Loss: 0.5003741979598999
Test Accuracy: 0.7835416793823242


In [64]:
df_test['cleaned_text'] = df_test['text'].apply(process_text)
df_test

,text,cleaned_text
0,Local soccer club ditch shirt sponsorship deal...,local soccer club ditch shirt sponsorship deal...
1,What is this weird metal thing in the mail fro...,weird metal thing mail publisher clearing hous...
2,Donald Trump Jr reportedly told in email of Ru...,donald trump reportedly told email russian gov...
3,Satanist prayer is held in Alaskan government ...,satanist prayer held alaskan government meetin...
4,There's A Spectacular Error In The Background ...,spectacular error background simpson episode m...
...,...,...
11950,Asteroid Impact Warning: ESA Says This NEO Wil...,asteroid impact warning esa say neo likely hit...
11951,President Donald Trump (right) during his inte...,president donald trump right interview special...
11952,Two Jewish prisoners caught while attempting t...,two jewish prisoner caught attempting escape a...
11953,Tool鈥檚 Maynard James Keenan Reveals Illness He...,tool鈥檚 maynard james keenan reveals illness he...


In [66]:
X_test_re = tokenizer.texts_to_sequences(df_test['cleaned_text'])
X_test_re = pad_sequences(X_test_re, maxlen=maxlen)


In [67]:
y_pred = model_rnn.predict(X_test_re)
y_pred_classes = np.argmax(y_pred, axis=1)


374/374 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [68]:
y_pred_classes

array([1, 1, 1, ..., 0, 0, 0], shape=(11955,))

In [79]:
submission = pd.DataFrame({
    'ID': range(1, len(y_pred_classes) + 1),
    'label': y_pred_classes
})
submission.to_csv('submission.csv', index=False)


In [80]:
submission

,ID,label
0,1,1
1,2,1
2,3,1
3,4,1
4,5,0
...,...,...
11950,11951,0
11951,11952,0
11952,11953,0
11953,11954,0
